# Register External Model Endpoint

Creates or validates the Databricks Model Serving external model endpoint used by RAG.
Default target: `access_drift_llm` → OpenAI `gpt-4o-mini`.

This notebook never accepts a plaintext API key. Store the key in a Databricks secret first.


In [ ]:
dbutils.widgets.text("endpoint_name", "access_drift_llm")
dbutils.widgets.text("model_name", "gpt-4o-mini")
dbutils.widgets.text("secret_scope", "access-drift-openai")
dbutils.widgets.text("secret_key", "api-key")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])
dbutils.widgets.dropdown("test_after_create", "true", ["true", "false"])

ENDPOINT_NAME = dbutils.widgets.get("endpoint_name").strip()
MODEL_NAME = dbutils.widgets.get("model_name").strip()
SECRET_SCOPE = dbutils.widgets.get("secret_scope").strip()
SECRET_KEY = dbutils.widgets.get("secret_key").strip()
DRY_RUN = dbutils.widgets.get("dry_run").lower() == "true"
TEST_AFTER_CREATE = dbutils.widgets.get("test_after_create").lower() == "true"

assert ENDPOINT_NAME, "endpoint_name is required"
assert MODEL_NAME, "model_name is required"
assert SECRET_SCOPE and SECRET_KEY, "secret_scope and secret_key are required"


In [ ]:
try:
    import mlflow.deployments
except ImportError:
    %pip install "mlflow[genai]>=2.9.0"
    dbutils.library.restartPython()


In [ ]:
import json
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

endpoint_config = {
    "served_entities": [
        {
            "name": "openai-gpt-4o-mini",
            "external_model": {
                "name": MODEL_NAME,
                "provider": "openai",
                "task": "llm/v1/chat",
                "openai_config": {
                    "openai_api_key": f"{{{{secrets/{SECRET_SCOPE}/{SECRET_KEY}}}}}",
                },
            },
        }
    ]
}

print("endpoint_name:", ENDPOINT_NAME)
print("config preview:")
print(json.dumps(endpoint_config, indent=2, ensure_ascii=False))

try:
    existing = client.get_endpoint(endpoint=ENDPOINT_NAME)
except Exception as exc:
    existing = None
    print(f"endpoint not found or not readable: {type(exc).__name__}: {exc}")

if existing:
    print(f"Endpoint already exists: {ENDPOINT_NAME}")
    display(existing)
elif DRY_RUN:
    print("dry_run=true: endpoint creation skipped")
else:
    created = client.create_endpoint(name=ENDPOINT_NAME, config=endpoint_config)
    print(f"Endpoint creation requested: {ENDPOINT_NAME}")
    display(created)


In [ ]:
if DRY_RUN or not TEST_AFTER_CREATE:
    print("test skipped")
else:
    response = client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={
            "messages": [
                {"role": "system", "content": "You are a concise security assistant."},
                {"role": "user", "content": "Return exactly: access-drift-llm-ok"},
            ],
            "temperature": 0,
            "max_tokens": 32,
        },
    )
    print(response["choices"][0]["message"]["content"])
